In [ ]:
# step 1. 저자 식별 체계 구축은 이 전에 완료

In [2]:
# step 2. TrendScore 산출 (키워드 표시 제거 버전)

import pandas as pd
import json
import os
import re

# =============================================================================
# 1. 파일 경로 설정
# =============================================================================
paper_json_path = 'SSU_Datathon2025_공학분야_62199_with_AUTR_ID.json'
author_name_csv = 'rep_rem_authors_in_korean.csv'
weights_root_folder = 'weights_by_year_class' 
society_rank_csv = 'society_rankings_corrected.csv' 
output_root_folder = 'TrendScore_Split_By_Year_Class'

if not os.path.exists(output_root_folder):
    os.makedirs(output_root_folder)

# =============================================================================
# 2. 데이터 로드: 학회 IF 점수 (Lookup Table)
# =============================================================================
print("1. 학회 IF 점수 데이터를 로드하여 매핑 테이블을 만듭니다...")

if_lookup = {} 

if os.path.exists(society_rank_csv):
    df_rank = pd.read_csv(society_rank_csv)
    for _, row in df_rank.iterrows():
        year = str(row['Year'])
        category = str(row['Category']).strip()
        ranking_text = str(row['Society_Rankings(IF)'])
        matches = re.findall(r'\d+\.\s+(.*?)\s+\(([\d\.]+)\)', ranking_text)
        for society_name, score in matches:
            key = (year, category, society_name.strip())
            if_lookup[key] = score
else:
    print("⚠️ 경고: society_rankings_corrected.csv 파일이 없습니다.")

# =============================================================================
# 3. 데이터 로드: 트렌드 가중치 & 저자명
# =============================================================================
print("2. 트렌드 가중치 및 저자 정보를 로드합니다...")

trend_lookup = {}
for root, dirs, files in os.walk(weights_root_folder):
    for file in files:
        if file.endswith("_weights.json"):
            try:
                with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                    data = json.load(f)
                items = data if isinstance(data, list) else [data]
                for item in items:
                    y = str(item.get('year', 'Unknown'))
                    c = item.get('class_name', 'Unknown')
                    for kw in item.get('keywords', []):
                        w = kw.get('weight', 0)
                        vs = kw.get('merged_variants', '')
                        variants = [v.strip().lower() for v in vs.split(';')] if vs else [kw.get('keyword','').strip().lower()]
                        for v in variants:
                            if v: trend_lookup[(y, c, v)] = w
            except: pass

id_to_name = {}
if os.path.exists(author_name_csv):
    df_names = pd.read_csv(author_name_csv)
    id_to_name = dict(zip(df_names['Author ID'].astype(str), df_names['Author Name']))

# =============================================================================
# 4. 분석 및 데이터 포맷팅
# =============================================================================
print("3. 연구자별 상세 내역 생성 중...")

with open(paper_json_path, 'r', encoding='utf-8') as f:
    paper_data = json.load(f)

author_stats = {}

for paper in paper_data.get("NODE_LIST", []):
    # 1) 기본 정보
    paper_id = paper.get("NODE_ID")
    pbsh = str(paper.get("PBSH", ""))[:4]
    class_name = paper.get("NODE_CLSS_02", "")
    iprd_nm = paper.get("IPRD_NM", "").strip()
    
    # 제목 추출
    paper_title = paper.get("NODE_TTLE", paper.get("NODE_TTLE_EN", "제목 없음")).strip()
    paper_title = paper_title.replace("\n", " ").replace("\r", "")

    if not pbsh or not class_name: continue

    # 2) 트렌드 점수 계산
    kwd_str = paper.get("KYWD", "")
    if not kwd_str: kwd_str = paper.get("KWD", "")
    keywords = [k.strip().lower() for k in kwd_str.split(",")] if kwd_str else []
    
    trend_score = 0
    for k in keywords:
        trend_score += trend_lookup.get((pbsh, class_name, k), 0)
    
    # 3) IF 점수 매핑
    if_score = if_lookup.get((pbsh, class_name, iprd_nm), "N/A")
    if_display = f"IF: {if_score}" if if_score != "N/A" else "IF: -"
    
    # [수정 완료] 상세 정보 문자열 생성 (키워드 라인 삭제)
    paper_info_str = (
        f"[{paper_title}]\n"
        f"   └ 학회: {iprd_nm} ({if_display})"
    )
    
    # 4) 저자별 할당
    autr_ids = paper.get("AUTR_ID", "")
    if autr_ids:
        for auth_id in autr_ids.split(","):
            if not auth_id: continue
            
            key = (pbsh, class_name, auth_id)
            
            if key not in author_stats:
                author_stats[key] = {
                    'Author_Name': id_to_name.get(auth_id, "Unknown"),
                    'Total_Trend_Score': 0,
                    'Papers': []
                }
            
            author_stats[key]['Total_Trend_Score'] += trend_score
            author_stats[key]['Papers'].append(paper_info_str)

# =============================================================================
# 5. 결과 저장
# =============================================================================
print("4. 결과 저장 중...")

final_rows = []
for (year, cls, auth_id), stat in author_stats.items():
    full_paper_text = "\n--------------------------------\n".join(stat['Papers'])
    
    final_rows.append({
        'Year': year,
        'Class_Name': cls,
        'Author_ID': auth_id,
        'Author_Name': stat['Author_Name'],
        'Total_Papers': len(stat['Papers']),
        'Total_Trend_Score': stat['Total_Trend_Score'],
        'Paper_List_Details': full_paper_text 
    })

df_final = pd.DataFrame(final_rows)

if df_final.empty:
    print("❌ 결과 데이터가 없습니다.")
else:
    grouped = df_final.groupby(['Year', 'Class_Name'])
    saved_count = 0
    
    for (year, class_name), group_df in grouped:
        year_folder = os.path.join(output_root_folder, str(year))
        if not os.path.exists(year_folder): os.makedirs(year_folder)
        
        sorted_df = group_df.sort_values(by=['Total_Trend_Score', 'Total_Papers'], ascending=[False, False])
        sorted_df.insert(0, 'Rank', range(1, len(sorted_df) + 1))
        
        safe_class = str(class_name).replace("/", "_").strip()
        filename = f"{year}_{safe_class}_Researcher_Analysis.csv"
        
        sorted_df.to_csv(os.path.join(year_folder, filename), index=False, encoding='utf-8-sig')
        saved_count += 1

    print(f"✅ 분석 완료! 총 {saved_count}개의 파일이 생성되었습니다.")
    print(f"📂 저장 위치: {output_root_folder}")
    print("👉 엑셀에서 파일을 열고 'Paper_List_Details' 셀의 [자동 줄 바꿈]을 켜시면 상세 내용이 잘 보입니다.")

1. 학회 IF 점수 데이터를 로드하여 매핑 테이블을 만듭니다...
2. 트렌드 가중치 및 저자 정보를 로드합니다...
3. 연구자별 상세 내역 생성 중...
4. 결과 저장 중...
✅ 분석 완료! 총 50개의 파일이 생성되었습니다.
📂 저장 위치: TrendScore_Split_By_Year_Class
👉 엑셀에서 파일을 열고 'Paper_List_Details' 셀의 [자동 줄 바꿈]을 켜시면 상세 내용이 잘 보입니다.


In [3]:
# step 3. FinalScore 산출 (논문 제목 표시 추가 버전)

import pandas as pd
import json
import os
import re

# =============================================================================
# 1. 파일 경로 설정 (사용자 로컬 환경 반영)
# =============================================================================
paper_json_path = 'SSU_Datathon2025_공학분야_62199_with_AUTR_ID.json'
author_name_csv = 'rep_rem_authors_in_korean.csv'
weights_root_folder = 'weights_by_year_class' 
society_rank_csv = 'society_rankings_corrected.csv' # IF 점수 참조 파일

# 결과 저장 폴더 (Final Score용)
output_root_folder = 'FinalScore_Researcher_Analysis_Detailed'

if not os.path.exists(output_root_folder):
    os.makedirs(output_root_folder)

# =============================================================================
# 2. 데이터 로드: 학회 IF 점수 (Lookup Table)
# =============================================================================
print("1. 학회 IF 점수 데이터를 로드하여 매핑 테이블을 만듭니다...")

if_lookup = {} # Key: (Year, Category, SocietyName), Value: IF_Score

if os.path.exists(society_rank_csv):
    df_rank = pd.read_csv(society_rank_csv)
    
    for _, row in df_rank.iterrows():
        year = str(row['Year'])
        category = str(row['Category']).strip()
        ranking_text = str(row['Society_Rankings(IF)'])
        
        matches = re.findall(r'\d+\.\s+(.*?)\s+\(([\d\.]+)\)', ranking_text)
        
        for society_name, score in matches:
            key = (year, category, society_name.strip())
            if_lookup[key] = float(score)
    print(f"   -> 총 {len(if_lookup)}개의 학회 IF 점수 매핑 정보를 확보했습니다.")
else:
    print(f"⚠️ 경고: {society_rank_csv} 파일이 없습니다.")

# =============================================================================
# 3. 데이터 로드: 트렌드 가중치 & 저자명
# =============================================================================
print("2. 트렌드 가중치 및 저자 정보를 로드합니다...")

# [트렌드 가중치]
trend_lookup = {}
for root, dirs, files in os.walk(weights_root_folder):
    for file in files:
        if file.endswith("_weights.json"):
            try:
                with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                    data = json.load(f)
                items = data if isinstance(data, list) else [data]
                for item in items:
                    y = str(item.get('year', 'Unknown'))
                    c = item.get('class_name', 'Unknown')
                    for kw in item.get('keywords', []):
                        w = kw.get('weight', 0)
                        vs = kw.get('merged_variants', '')
                        variants = [v.strip().lower() for v in vs.split(';')] if vs else [kw.get('keyword','').strip().lower()]
                        for v in variants:
                            if v: trend_lookup[(y, c, v)] = w
            except: pass

# [저자 이름]
id_to_name = {}
if os.path.exists(author_name_csv):
    df_names = pd.read_csv(author_name_csv)
    id_to_name = dict(zip(df_names['Author ID'].astype(str), df_names['Author Name']))

# =============================================================================
# 4. 분석 및 데이터 포맷팅 (Final Score 산출)
# =============================================================================
print("3. 연구자별 Final Score 산출 및 상세 내역(제목 포함) 생성 중...")

with open(paper_json_path, 'r', encoding='utf-8') as f:
    paper_data = json.load(f)

# 저자별 데이터를 모을 딕셔너리
author_stats = {}

for paper in paper_data.get("NODE_LIST", []):
    # 1) 기본 정보 추출
    paper_id = paper.get("NODE_ID")
    pbsh = str(paper.get("PBSH", ""))[:4]
    class_name = paper.get("NODE_CLSS_02", "")
    iprd_nm = paper.get("IPRD_NM", "").strip() # 학회명

    # [수정] 논문 제목 추출 (국문 우선 -> 영문 -> 없음)
    paper_title = paper.get("NODE_TTLE", paper.get("NODE_TTLE_EN", "제목 없음")).strip()
    paper_title = paper_title.replace("\n", " ").replace("\r", "")
    
    if not pbsh or not class_name: continue

    # 2) 트렌드 점수 (TrendScore) 계산
    kwd_str = paper.get("KYWD", "")
    if not kwd_str: kwd_str = paper.get("KWD", "")
    keywords = [k.strip().lower() for k in kwd_str.split(",")] if kwd_str else []
    
    trend_score = 0
    for k in keywords:
        trend_score += trend_lookup.get((pbsh, class_name, k), 0)
    
    # 3) IF 점수 (Quality Score) 적용
    if_score = if_lookup.get((pbsh, class_name, iprd_nm), 1.0)
    
    # [FinalScore 수식 적용: 트렌드 점수 * IF 점수]
    final_paper_score = trend_score * if_score
    
    # 4) 상세 내역 문자열 생성 (제목 포함)
    if (pbsh, class_name, iprd_nm) in if_lookup:
        if_display = f"IF: {if_score}"
    else:
        if_display = "IF: -" # 랭킹에 없는 학회 (기본값 1.0)
    
    # [수정] 포맷 변경: 제목 + 학회 정보
    paper_info_str = (
        f"[{paper_title}]\n"
        f"   └ 학회: {iprd_nm} ({if_display})"
    )
    
    # 5) 저자별 데이터 집계
    autr_ids = paper.get("AUTR_ID", "")
    if autr_ids:
        for auth_id in autr_ids.split(","):
            if not auth_id: continue
            
            key = (pbsh, class_name, auth_id)
            if key not in author_stats:
                author_stats[key] = {
                    'Author_Name': id_to_name.get(auth_id, "Unknown"),
                    'Total_Final_Score': 0.0,
                    'Raw_Trend_Score': 0.0,
                    'Papers': []
                }
            
            # 점수 누적
            author_stats[key]['Total_Final_Score'] += final_paper_score
            author_stats[key]['Raw_Trend_Score'] += trend_score
            author_stats[key]['Papers'].append(paper_info_str)

# =============================================================================
# 5. 결과 저장 (줄바꿈 적용)
# =============================================================================
print("4. 결과 저장 중...")

final_rows = []
for (year, cls, auth_id), stat in author_stats.items():
    # [수정] 논문 간 구분선 추가
    full_paper_text = "\n--------------------------------\n".join(stat['Papers'])
    
    final_rows.append({
        'Year': year,
        'Class_Name': cls,
        'Author_ID': auth_id,
        'Author_Name': stat['Author_Name'],
        'Total_Papers': len(stat['Papers']),
        'Final_Score': round(stat['Total_Final_Score'], 4), # 소수점 정리
        'Raw_Trend_Score': stat['Raw_Trend_Score'], # 참고용
        'Paper_List_Details': full_paper_text
    })

df_final = pd.DataFrame(final_rows)

if df_final.empty:
    print("❌ 결과 데이터가 없습니다.")
else:
    grouped = df_final.groupby(['Year', 'Class_Name'])
    saved_count = 0
    
    for (year, class_name), group_df in grouped:
        year_folder = os.path.join(output_root_folder, str(year))
        if not os.path.exists(year_folder): os.makedirs(year_folder)
        
        # Final Score 기준 내림차순 정렬
        sorted_df = group_df.sort_values(by=['Final_Score', 'Total_Papers'], ascending=[False, False])
        
        # 순위(Rank) 추가
        sorted_df.insert(0, 'Rank', range(1, len(sorted_df) + 1))
        
        # 파일명 생성
        safe_class = str(class_name).replace("/", "_").strip()
        filename = f"{year}_{safe_class}_Final_Analysis.csv"
        
        sorted_df.to_csv(os.path.join(year_folder, filename), index=False, encoding='utf-8-sig')
        saved_count += 1

    print(f"✅ 분석 완료! 총 {saved_count}개의 파일이 생성되었습니다.")
    print(f"📂 저장 위치: {output_root_folder}")
    print("👉 엑셀에서 파일을 열고 'Paper_List_Details' 셀의 [자동 줄 바꿈]을 켜시면 상세 내역이 보입니다.")

1. 학회 IF 점수 데이터를 로드하여 매핑 테이블을 만듭니다...
   -> 총 531개의 학회 IF 점수 매핑 정보를 확보했습니다.
2. 트렌드 가중치 및 저자 정보를 로드합니다...
3. 연구자별 Final Score 산출 및 상세 내역(제목 포함) 생성 중...
4. 결과 저장 중...
✅ 분석 완료! 총 50개의 파일이 생성되었습니다.
📂 저장 위치: FinalScore_Researcher_Analysis_Detailed
👉 엑셀에서 파일을 열고 'Paper_List_Details' 셀의 [자동 줄 바꿈]을 켜시면 상세 내역이 보입니다.


In [4]:
# step 4. FinalScore을 토대로 연도x중분류 별 Top 10 산출 (상세 버전)

import pandas as pd
import os

# =============================================================================
# 1. 파일 경로 설정
# =============================================================================
input_root_folder = 'FinalScore_Researcher_Analysis_Detailed' # Step 3 결과 폴더
output_file_name = 'Total_Top10_Summary_Detailed.csv' # 최종 요약 파일

# =============================================================================
# 2. Top 10 추출 및 포맷팅
# =============================================================================
print("연도별/중분류별 Top 10 연구자 상세 리스트를 생성합니다...")

summary_results = []

if not os.path.exists(input_root_folder):
    print(f"❌ 오류: 입력 폴더 '{input_root_folder}'를 찾을 수 없습니다. Step 3를 먼저 실행해주세요.")
else:
    # 폴더 순회
    for root, dirs, files in os.walk(input_root_folder):
        for file in files:
            if file.endswith("_Final_Analysis.csv"):
                file_path = os.path.join(root, file)
                
                try:
                    df = pd.read_csv(file_path)
                    
                    if df.empty: continue
                    
                    # 1) Final_Score 기준 내림차순 정렬 후 상위 10명 추출
                    top10_df = df.sort_values(by='Final_Score', ascending=False).head(10)
                    
                    # 2) 순위(Rank) 재부여 (1위~10위)
                    top10_df['Rank'] = range(1, len(top10_df) + 1)
                    
                    # 3) 필요한 컬럼만 선택해서 결과 리스트에 추가
                    # Step 3에서 생성된 모든 상세 정보를 그대로 가져옵니다.
                    for _, row in top10_df.iterrows():
                        summary_results.append({
                            'Year': row['Year'],
                            'Class_Name': row['Class_Name'],
                            'Rank': row['Rank'],
                            'Author_Name': row['Author_Name'],
                            'Final_Score': row['Final_Score'],
                            'Total_Papers': row['Total_Papers'],
                            'Paper_List_Details': row['Paper_List_Details'] # 상세 내역(제목, 학회 등)
                        })
                    
                except Exception as e:
                    print(f"파일 처리 중 오류 발생 ({file}): {e}")

# =============================================================================
# 3. 결과 저장
# =============================================================================
if summary_results:
    summary_df = pd.DataFrame(summary_results)
    
    # [정렬 기준]
    # 1. 연도 (오름차순) -> 2021, 2022...
    # 2. 중분류 (오름차순) -> 건축, 기계...
    # 3. 순위 (오름차순) -> 1위, 2위...
    summary_df = summary_df.sort_values(by=['Year', 'Class_Name', 'Rank'], ascending=[True, True, True])
    
    # 저장
    summary_df.to_csv(output_file_name, index=False, encoding='utf-8-sig')
    
    print("-" * 50)
    print(f"✅ 작업 완료! '{output_file_name}' 파일이 생성되었습니다.")
    print("-" * 50)
    print("👉 [엑셀 팁]")
    print("1. 파일을 엽니다.")
    print("2. 'Paper_List_Details' 컬럼의 [자동 줄 바꿈]을 켭니다.")
    print("3. 가독성을 위해 'Year'와 'Class_Name' 컬럼을 선택 후 [병합하고 가운데 맞춤]을 하시면 좋습니다.")
else:
    print("❌ 처리할 데이터가 없습니다.")

연도별/중분류별 Top 10 연구자 상세 리스트를 생성합니다...
--------------------------------------------------
✅ 작업 완료! 'Total_Top10_Summary_Detailed.csv' 파일이 생성되었습니다.
--------------------------------------------------
👉 [엑셀 팁]
1. 파일을 엽니다.
2. 'Paper_List_Details' 컬럼의 [자동 줄 바꿈]을 켭니다.
3. 가독성을 위해 'Year'와 'Class_Name' 컬럼을 선택 후 [병합하고 가운데 맞춤]을 하시면 좋습니다.
